In [1]:
import re
import os
import time
import random
import cloudscraper

import pandas as pd

from tqdm import tqdm
from bs4 import BeautifulSoup

In [2]:
url = 'https://awoiaf.westeros.org/index.php/List_of_characters'
domain = 'https://awoiaf.westeros.org'

In [3]:
scraper = cloudscraper.create_scraper()
response = scraper.get(url)

In [4]:
print(response.status_code)
if response.status_code == 200:
    text = response.text
    soup = BeautifulSoup(text, 'html5lib')

200


In [5]:
len(soup.find_all('li'))

3754

In [6]:
character_name = []
webpage = []

for li in soup.find_all('li'):
    a_tag = li.find('a')
    if set(a_tag.attrs.keys()) == {'title', 'href'}:
        name = a_tag['title']
        if ':' not in name:
            character_name.append(name)
            webpage.append(domain + a_tag['href'])

In [7]:
df = pd.DataFrame({'character_name': character_name, 'webpage': webpage})
df.head()

,character_name,webpage
0,A certain man,https://awoiaf.westeros.org/index.php/A_certai...
1,Abelar Hightower,https://awoiaf.westeros.org/index.php/Abelar_H...
2,Abelon,https://awoiaf.westeros.org/index.php/Abelon
3,Addam of Duskendale,https://awoiaf.westeros.org/index.php/Addam_of...
4,Addam Frey,https://awoiaf.westeros.org/index.php/Addam_Frey


In [8]:
print(f"Total potential characters found: {len(df)}")

Total potential characters found: 3604


### Aliases recevoring on one character

In [9]:
url = "https://awoiaf.westeros.org/index.php/Arya_Stark"

In [10]:
response = scraper.get(url)

In [11]:
print(response.status_code)
if response.status_code == 200:
    text = response.text
    soup = BeautifulSoup(text, 'html5lib')

200


In [12]:
soup.find_all('th')

[<th colspan="2" style="text-align:center;font-size:125%;font-weight:bold;font-weight:bold; background: none; line-height: 1.1em;"><table style="width:100%"><tbody><tr><td style="border:none;vertical-align:top;text-align:left;width:50px"><a href="/index.php/House_Stark" title="House Stark"><img alt="House Stark.svg" height="55" src="/images/thumb/7/7e/House_Stark.svg/50px-House_Stark.svg.png" srcset="/images/thumb/7/7e/House_Stark.svg/75px-House_Stark.svg.png 1.5x, /images/thumb/7/7e/House_Stark.svg/100px-House_Stark.svg.png 2x" width="50"/></a></td><td style="border:none;vertical-align:middle;text-align:center;padding:0px 7px 5px 7px">Arya Stark</td><td style="border:none;vertical-align:top;text-align:right;width:50px"><a href="/index.php/Faceless_Men" title="Faceless Men"><img alt="Faceless Men.svg" height="50" src="/images/thumb/a/a3/Faceless_Men.svg/50px-Faceless_Men.svg.png" srcset="/images/thumb/a/a3/Faceless_Men.svg/75px-Faceless_Men.svg.png 1.5x, /images/thumb/a/a3/Faceless_Men

In [13]:
aliases = []
for th in soup.find_all('th'):
    if th.text.strip() == 'Aliases':
        names_list = th.find_next_sibling('td').text.strip()
        names_list = [name.strip() for name in re.split(r'(?:\[\d+\])+', names_list) if name.strip()]
        aliases.append(names_list)

print(aliases)

[['Arya Horseface', 'Arya Underfoot', 'Arry', 'Lumpyhead', 'Lumpyface', 'Stickboy', 'Rabbitkiller', 'Weasel', 'The ghost in Harrenhal', 'Nymeria', 'Nan', 'Squab', 'Squirrel', 'Wolf girl', 'Salty', 'Cat of the Canals', 'Blind Beth', 'The blind girl', 'The night wolf', 'The ugly girl', 'Mercedene', 'Mercy']]


### Website Mirroring

In [ ]:
# Ensure the folder exists
if not os.path.exists('awoiaf_pages'):
    os.makedirs('awoiaf_pages')

# Using iterrows so we can access both 'character_name' and 'webpage'
for index, row in tqdm(df.iterrows(), total=len(df), desc="Mirroring website", unit="page"):
    character = str(row['character_name'])
    url = row['webpage']
    
    # Standardize the URL: ensure it's a full link if the df only has relative paths
    if not url.startswith('http'):
        url = domain + url

    # Generate a safe filename based on the character name
    filename = f"awoiaf_pages/{character.replace(' ', '_').replace('/', '-')}.html"
    
    # Skip if we already have it local
    if os.path.exists(filename):
        continue
        
    try:
        # Increased timeout and scraper request
        response = scraper.get(url, timeout=30)
        
        if response.status_code == 200:
            with open(filename, 'w', encoding='utf-8') as f:
                f.write(response.text)
        elif response.status_code == 404:
            print(f"\n[!] 404 Not Found: {url}")
        elif response.status_code == 403:
            print("\n[!] 403 Forbidden: You might be going too fast!")
            break # Stop immediately if banned
        
        # Safety politeness delay
        time.sleep(random.uniform(1, 3))
        
    except Exception as e:
        print(f"\nError downloading {character} from {url}: {e}")

Mirroring website: 100%|██████████| 3604/3604 [00:05<00:00, 705.56page/s] 


#### Verify the integrity of the html pages

In [15]:
import os
import re
from tqdm import tqdm

folder = 'awoiaf_pages'

# 1. Gather the two sets for comparison
# Found: What is actually on your disk
found_files = {f for f in os.listdir(folder) if f.endswith('.html')}

# Expected: What SHOULD be there based on your DataFrame
expected_files = {
    f"{char.replace(' ', '_').replace('/', '-')}.html" 
    for char in df['character_name']
}

# 2. Calculate the Differences
missing_files = sorted(list(expected_files - found_files))   # In DF but not in folder
unexpected_files = sorted(list(found_files - expected_files)) # In folder but not in DF

# 3. Audit Integrity (Only for files that exist)
empty_files = []
truncated_files = []

for file in tqdm(found_files, desc="Auditing HTML integrity", unit="file"):
    path = os.path.join(folder, file)
    try:
        size = os.path.getsize(path)
        if size == 0:
            empty_files.append(file)
            continue
        
        with open(path, 'r', encoding='utf-8') as f:
            if size > 100:
                f.seek(size - 100)
            else:
                f.seek(0)
            
            last_chunk = f.read()
            if '</html>' not in last_chunk.lower():
                truncated_files.append(file)
    except Exception as e:
        tqdm.write(f"Error reading {file}: {e}")

# --- Comparison & Audit Report ---
print(f"\n" + "="*30)
print(f"      FILE SYSTEM AUDIT")
print(f"="*30)
print(f"Total Expected (DF):    {len(expected_files)}")
print(f"Total Found (Folder):   {len(found_files)}")
print(f"---")
print(f"✅ Matches:             {len(expected_files & found_files)}")
print(f"❌ Missing:             {len(missing_files)}")
print(f"❓ Unexpected/Extra:    {len(unexpected_files)}")
print(f"---")
print(f"📁 Empty Files:         {len(empty_files)}")
print(f"✂️  Truncated Files:     {len(truncated_files)}")

# 4. Detailed Breakdown
if missing_files:
    print(f"\n[!] MISSING ({len(missing_files)} files):")
    for f in missing_files[:5]:
        print(f"  - {f}")
    if len(missing_files) > 5: print(f"    ... and {len(missing_files)-5} more.")

if unexpected_files:
    print(f"\n[?] UNEXPECTED/EXTRA ({len(unexpected_files)} files):")
    print("Check if these are old downloads or named differently in the DF.")
    for f in unexpected_files[:5]:
        print(f"  - {f}")
    if len(unexpected_files) > 5: print(f"    ... and {len(unexpected_files)-5} more.")

if empty_files or truncated_files:
    print(f"\n[!] INTEGRITY ISSUES:")
    for f in (empty_files + truncated_files)[:5]:
        print(f"  - {f}")

Auditing HTML integrity: 100%|██████████| 3594/3594 [16:08<00:00,  3.71file/s]


      FILE SYSTEM AUDIT
Total Expected (DF):    3595
Total Found (Folder):   3594
---
✅ Matches:             3594
❌ Missing:             1
❓ Unexpected/Extra:    0
---
📁 Empty Files:         0
✂️  Truncated Files:     0

[!] MISSING (1 files):
  - Hooded_Man.html


### Aliases recovering on all characters

In [16]:
local_data = []

for character in tqdm(df['character_name'], desc="Extracting Aliases", unit="char"):
    # Match the filename format used during your download step
    filename = f"awoiaf_pages/{character.replace(' ', '_').replace('/', '-')}.html"
    
    found_alias = "None"
    
    if os.path.exists(filename):
        try:
            with open(filename, 'r', encoding='utf-8') as f:
                html_content = f.read()
                
            soup = BeautifulSoup(html_content, 'html5lib')
            
            # Look for the Table Header 'Aliases'
            for th in soup.find_all('th'):
                if th.get_text(strip=True) == 'Aliases':
                    td = th.find_next_sibling('td')
                    if td:
                        # Get text with a space separator to prevent words from sticking together
                        # raw_text = td.get_text(separator=" ", strip=True)
                        raw_text = th.find_next_sibling('td').text.strip()
                        
                        # Apply the regex to split by [1], [2], [10][11], etc.
                        # This removes citations and returns a clean list of strings
                        names_list = [name.strip() for name in re.split(r'(?:\[\d+\])+', raw_text) if name.strip()]
                        
                        # Store the list
                        if names_list:
                            found_alias = names_list
                    break # Exit the 'th' loop once we found our target
        except Exception as e:
            # Use tqdm.write so the progress bar doesn't break if an error occurs
            tqdm.write(f"Error processing {character}: {e}")
            found_alias = "Error"
            
        local_data.append(found_alias)
        
    else:
        # If the file wasn't downloaded in the previous step
        local_data.append("File Missing")

df['aliases'] = local_data

print("\n--- Extraction Complete ---")
print(df[['character_name', 'aliases']].head(10))



Extracting Aliases: 100%|██████████| 3604/3604 [1:03:16<00:00,  1.05s/char]


--- Extraction Complete ---
        character_name aliases
0        A certain man    None
1     Abelar Hightower    None
2               Abelon    None
3  Addam of Duskendale    None
4           Addam Frey    None
5      Addam Hightower    None
6       Addam Marbrand    None
7         Addam Osgrey    None
8         Addam Rivers    None
9       Addam Velaryon    None


In [17]:
df.head()

,character_name,webpage,aliases
0,A certain man,https://awoiaf.westeros.org/index.php/A_certai...,None
1,Abelar Hightower,https://awoiaf.westeros.org/index.php/Abelar_H...,None
2,Abelon,https://awoiaf.westeros.org/index.php/Abelon,None
3,Addam of Duskendale,https://awoiaf.westeros.org/index.php/Addam_of...,None
4,Addam Frey,https://awoiaf.westeros.org/index.php/Addam_Frey,None


In [18]:
df.to_pickle('./data/scrapped_characters.pickle')